In [1]:
from src.aggregation.agg_spark import ElectricityAggregator


import os
import pandas as pd
from datetime import timedelta
import matplotlib.pyplot as plt
import seaborn as sns   
import pyarrow.parquet as pq
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# data paths
DATA_DIR = "mock_data"
OUT_BASE_DIR = "output"

MATDATA_PATH = os.path.join(DATA_DIR, "meter_data_500.snappy.parquet")
TARIFF_PATH  = os.path.join(DATA_DIR, "tariff_100.csv")
SURVEY_PATH  = os.path.join(DATA_DIR, "survey_50.csv")

# Electricity meter data
pf = pq.ParquetFile(MATDATA_PATH)
table = pf.read(columns=["aID", "TIDPUNKT", "TIDPUNKT_DAG", "FORBRUKNING_KWH"])
meter_data = table.to_pandas()

print("Unique households:", len(meter_data["aID"].unique())) # unique households
print("Unique days:", len(meter_data["TIDPUNKT_DAG"].unique())) # unique days
display(meter_data.head(3))

# Tariff data
tariff_df = pd.read_csv(TARIFF_PATH, sep=",")
print(f"tariff rows: {len(tariff_df):,}") # Tariff number 
display(tariff_df.head(3))

# Survey data
# survey_df = pd.read_csv(SURVEY_PATH)
# print(f"survey rows: {len(survey_df):,}") # Survey number
# display(survey_df.head(3))


Unique households: 500
Unique days: 782


,aID,TIDPUNKT,TIDPUNKT_DAG,FORBRUKNING_KWH
0,735999166200000851,2024-01-01 00:00:00+00:00,2024-01-01,0.283425
1,735999166200000851,2024-01-01 01:00:00+00:00,2024-01-01,0.300204
2,735999166200000851,2024-01-01 02:00:00+00:00,2024-01-01,0.260809


tariff rows: 100


,Produktnamn,Startdatum,GS1-nr
0,GENAB Tidsindelad 6 kW Villa,2025-06-01,735999166200288372
1,GENAB Tidsindelad 6 kW Villa,2025-04-01,735999166200186244
2,GENAB Tidsindelad 6 kW Villa,2025-03-01,735999166200055594


In [3]:
agg = ElectricityAggregator(meter_data, tariff_df)

monthly = agg.run(
    freq="month",
    agg_method="top3_mean"
)

display(monthly)

Initializing ElectricityAggregator...


AttributeError: 'DataFrame' object has no attribute 'withColumn'

In [ ]:
# Week & Weekend
weekday_result = agg.run(
    freq="weekday",
    agg_method=["mean"],
    use_price=True,
    add_user_group_col=[0.25, 0.75],
    )


Electricity aggregation start
Rows: 9,372,500
Frequency: weekday
Aggregation: ['mean']
Include price split (all/high/low): True
Add User groups: [0.25, 0.75]
Creating usage groups...
Usage groups created (3.58s)
Aggregating per household...
Aggregation done (8.41s)
Total runtime: 11.99s


In [ ]:
weekday_result

,aID,TIDPUNKT,price,tariff_active,mean_consumption,tariff_start,tariff_plan,usage_group
0,735999166200000851,0,all,0,0.334415,NaT,NaN,low
1,735999166200000851,1,all,0,0.333322,NaT,NaN,low
2,735999166200000851,2,all,0,0.333210,NaT,NaN,low
3,735999166200000851,3,all,0,0.331530,NaT,NaN,low
4,735999166200000851,4,all,0,0.332026,NaT,NaN,low
...,...,...,...,...,...,...,...,...
11395,735999166200499234,3,high,0,0.421502,NaT,NaN,medium
11396,735999166200499234,4,low,0,0.385311,NaT,NaN,medium
11397,735999166200499234,4,high,0,0.420283,NaT,NaN,medium
11398,735999166200499234,5,low,0,0.425663,NaT,NaN,medium
